In [1]:
import yfinance as yf
import pandas as pd
import datetime as d
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings(action='ignore')

In [2]:
s=d.datetime(2020,1,1)
e=d.datetime(2026,4,19)

In [3]:
tcs=yf.download('TCS.ns', start=s, end=e)
tcs.columns=tcs.columns.get_level_values(0)
tcs['Stock'] = 'TCS' 

[*********************100%***********************]  1 of 1 completed


In [4]:
wipro=yf.download('Wipro.ns', start=s, end=e)
wipro.columns=wipro.columns.get_level_values(0)
wipro['Stock']='Wipro'

[*********************100%***********************]  1 of 1 completed


In [5]:
infosys=yf.download('INFY.ns', start=s, end=e)
infosys.columns=infosys.columns.get_level_values(0)
infosys['Stock']='INFY'

[*********************100%***********************]  1 of 1 completed


In [6]:
hcltech=yf.download('HCLTECH.ns', start=s, end=e)
hcltech.columns=hcltech.columns.get_level_values(0)
hcltech['Stock']='HCLTECH'

[*********************100%***********************]  1 of 1 completed


In [7]:
techmahindra =yf.download('TECHM.ns', start=s, end=e)
techmahindra.columns=techmahindra.columns.get_level_values(0)
techmahindra['Stock']='TECHM'

[*********************100%***********************]  1 of 1 completed


In [8]:
ltimindtree =yf.download('LTIM.ns', start=s, end=e)
ltimindtree.columns=ltimindtree.columns.get_level_values(0)
ltimindtree['Stock']='LTIM'

[*********************100%***********************]  1 of 1 completed


In [9]:
persistent =yf.download('PERSISTENT.ns', start=s, end=e)
persistent.columns=persistent.columns.get_level_values(0)
persistent['Stock']='PERSISTENT'

[*********************100%***********************]  1 of 1 completed


In [10]:
oracle =yf.download('OFSS.ns', start=s, end=e)
oracle.columns=oracle.columns.get_level_values(0)
oracle['Stock']='OFSS'

[*********************100%***********************]  1 of 1 completed


In [11]:
coforge =yf.download('COFORGE.ns', start=s, end=e)
coforge.columns=coforge.columns.get_level_values(0)
coforge['Stock']='COFORGE'

[*********************100%***********************]  1 of 1 completed


In [12]:
mphasis =yf.download('MPHASIS.ns', start=s, end=e)
mphasis.columns=mphasis.columns.get_level_values(0)
mphasis['Stock']='MPHASIS'

[*********************100%***********************]  1 of 1 completed


In [13]:
df=pd.concat([tcs, wipro, infosys, hcltech, techmahindra, ltimindtree, persistent, oracle, coforge, mphasis], axis=0)

In [14]:
df

Price,Close,High,Low,Open,Volume,Stock
Date,,,,,,
2020-01-01,1866.113770,1880.146473,1854.405277,1866.458050,1354908,TCS
2020-01-02,1857.547852,1876.746243,1850.273183,1876.746243,2380752,TCS
2020-01-03,1894.566895,1913.808372,1863.014537,1863.014537,4655761,TCS
2020-01-06,1894.394897,1916.348167,1883.590403,1898.312091,3023209,TCS
2020-01-07,1899.043823,1906.619685,1880.060662,1894.437858,2429317,TCS
...,...,...,...,...,...,...
2026-04-10,2326.699951,2415.100098,2276.199951,2415.100098,1026126,MPHASIS
2026-04-13,2316.100098,2335.600098,2270.399902,2295.000000,229386,MPHASIS
2026-04-15,2409.300049,2414.800049,2345.699951,2375.000000,1011941,MPHASIS


In [15]:
df.to_csv("IT_Stock_NSE.csv")

In [ ]:
print(df.Stock.unique())
e = input("Enter Stock Name: ")
t = df[df.Stock == e]
t

['TCS' 'Wipro' 'INFY' 'HCLTECH' 'TECHM' 'LTIM' 'PERSISTENT' 'OFSS'
 'COFORGE' 'MPHASIS']


In [ ]:
t=t[['Close']]
t['Returns']=t['Close'].pct_change()
t.dropna(inplace=True)

In [ ]:
t

In [ ]:
#step 3: stationarity check using Augmented Dickey-Fuller (ADF) test
def check_stationarity(timeseries):
    result= adfuller(timeseries.dropna())
    print(f"ADF Statistic: {result[0]}")
    print(f"p-value: {result[1]}")
    
    if result[1]<0.05:
        print("The series is stationary.")
    else:
        print("The series is NOT stationary")

In [ ]:
check_stationarity(t['Close'])

In [ ]:
t['Close_Diff']=t['Close'].diff().dropna()
check_stationarity(t['Close_Diff'])

In [ ]:
model=ARIMA(t['Close'], order=(5,1,0)) # AR(5), I(1), MA(0)
model_fit= model.fit()

In [ ]:
forecasting = model_fit.forecast(steps=20)
dates=pd.date_range(start=t.index[-1], periods=21, freq='B')[1:]

In [ ]:
plt.figure(figsize=(10,5))
plt.xticks(rotation=90)
plt.plot(t['Close'], label="Actual Prices")
plt.plot(dates, forecasting, label="Predicted Prices", linestyle="dashed", color="red")
plt.show()